# Band-Split VAE Training (Colab)

Notebook version of `train_bandvae.py` with Google Drive support.

In [ ]:

# Mount Google Drive when on Colab
import os
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/liveness_detection_vae')
except ModuleNotFoundError:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Working directory: {PROJECT_ROOT}")


In [ ]:

import time
from pathlib import Path

import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from config_bandvae import get_config
from dataset_bandvae import LipLivenessBandDataset
from model_bandvae import BandSplitVAE, band_split_vae_loss

DATA_ROOT = Path('/content/drive/MyDrive/20GBprocessed/processed_live') if 'google.colab' in sys.modules else Path('20GBprocessed/processed_live')
SAVE_DIR = PROJECT_ROOT / 'runs' / 'bandvae_colab'

config = get_config('simple')
config.data_dir = str(DATA_ROOT)
config.save_dir = str(SAVE_DIR)
config.device = 'mps' if torch.backends.mps.is_available() else 'cpu'
config.num_workers = 2
config.pin_memory = False
config.epochs = 2  # adjust for full training

print(config)


In [ ]:

            def train_epoch(model, loader, optimizer, cfg, epoch):
                model.train()
                total_loss = recon_lf = recon_bp = recon_hf = kl_lf = kl_bp = kl_hf = 0.0
                n_batches = len(loader)
                start = time.time()

                for batch_idx, (x_lf, x_bp, x_hf) in enumerate(loader):
                    x_lf = x_lf.to(cfg.device)
                    x_bp = x_bp.to(cfg.device)
                    x_hf = x_hf.to(cfg.device)

                    recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
                    targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
                    betas = {'lf': cfg.beta_lf, 'bp': cfg.beta_bp, 'hf': cfg.beta_hf}
                    loss, loss_dict = band_split_vae_loss(
                        recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
                    )

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    total_loss += loss_dict['total']
                    recon_lf += loss_dict['recon_lf']
                    recon_bp += loss_dict['recon_bp']
                    recon_hf += loss_dict['recon_hf']
                    kl_lf += loss_dict['kl_lf']
                    kl_bp += loss_dict['kl_bp']
                    kl_hf += loss_dict['kl_hf']

                    if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == n_batches:
                        elapsed = time.time() - start
                        avg_time = elapsed / (batch_idx + 1)
                        eta = (n_batches - (batch_idx + 1)) * avg_time
                        n = batch_idx + 1
                        total_recon = (recon_lf + recon_bp + recon_hf) / n
                        total_kl = (kl_lf + kl_bp + kl_hf) / n
                        print(
                            f"  [Epoch {epoch:02d} | {n:04d}/{n_batches}] Total={total_loss / n:.4f} "
                            f"(Recon={total_recon:.4f} KL={total_kl:.4f}) | "
                            f"LF:[R={recon_lf / n:.4f} K={kl_lf / n:.4f}] "
                            f"BP:[R={recon_bp / n:.4f} K={kl_bp / n:.4f}] "
                            f"HF:[R={recon_hf / n:.4f} K={kl_hf / n:.4f}] | "
                            f"ETA={eta / 60:.1f}m"
                        )

                return {
                    'total': total_loss / n_batches,
                    'recon_lf': recon_lf / n_batches,
                    'recon_bp': recon_bp / n_batches,
                    'recon_hf': recon_hf / n_batches,
                    'kl_lf': kl_lf / n_batches,
                    'kl_bp': kl_bp / n_batches,
                    'kl_hf': kl_hf / n_batches,
                }


            @torch.no_grad()
            def validate(model, loader, cfg):
                model.eval()
                total_loss = recon_lf = recon_bp = recon_hf = kl_lf = kl_bp = kl_hf = 0.0
                n_samples = 0
                for x_lf, x_bp, x_hf in loader:
                    x_lf = x_lf.to(cfg.device)
                    x_bp = x_bp.to(cfg.device)
                    x_hf = x_hf.to(cfg.device)

                    recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
                    targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
                    betas = {'lf': cfg.beta_lf, 'bp': cfg.beta_bp, 'hf': cfg.beta_hf}
                    _, loss_dict = band_split_vae_loss(
                        recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
                    )

                    batch_size = x_lf.size(0)
                    total_loss += loss_dict['total'] * batch_size
                    recon_lf += loss_dict['recon_lf'] * batch_size
                    recon_bp += loss_dict['recon_bp'] * batch_size
                    recon_hf += loss_dict['recon_hf'] * batch_size
                    kl_lf += loss_dict['kl_lf'] * batch_size
                    kl_bp += loss_dict['kl_bp'] * batch_size
                    kl_hf += loss_dict['kl_hf'] * batch_size
                    n_samples += batch_size

                return {
                    'total': total_loss / n_samples,
                    'recon_lf': recon_lf / n_samples,
                    'recon_bp': recon_bp / n_samples,
                    'recon_hf': recon_hf / n_samples,
                    'kl_lf': kl_lf / n_samples,
                    'kl_bp': kl_bp / n_samples,
                    'kl_hf': kl_hf / n_samples,
                }


            def run_training(cfg):
                data_dir = Path(cfg.data_dir)
                if not data_dir.exists():
                    raise FileNotFoundError(f"Data directory not found: {data_dir}")

                save_dir = Path(cfg.save_dir)
                save_dir.mkdir(parents=True, exist_ok=True)

                print('=' * 60)
                print('Band-Split VAE Training (Colab)')
                print('=' * 60)
                print(cfg)
                print(f"Save dir: {save_dir}
")

                dataset = LipLivenessBandDataset(
                    data_dir=cfg.data_dir,
                    T_fixed=cfg.T_fixed,
                    fps=cfg.fps,
                    use_procrustes=True,
                    use_acceleration=cfg.use_acceleration,
                    use_angle=cfg.use_angle,
                    use_angle_rate=cfg.use_angle_rate,
                    fc_low=cfg.fc_low,
                    fc_high=cfg.fc_high,
                    filter_order=cfg.filter_order,
                )

                train_size = int((1 - cfg.val_split) * len(dataset))
                val_size = len(dataset) - train_size
                train_dataset, val_dataset = random_split(
                    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
                )

                train_loader = DataLoader(
                    train_dataset,
                    batch_size=cfg.batch_size,
                    shuffle=True,
                    num_workers=cfg.num_workers,
                    pin_memory=cfg.pin_memory,
                )

                val_loader = DataLoader(
                    val_dataset,
                    batch_size=cfg.batch_size,
                    shuffle=False,
                    num_workers=cfg.num_workers,
                    pin_memory=cfg.pin_memory,
                )

                model = BandSplitVAE(
                    C_in_per_band=cfg.C_in_per_band,
                    C_h=cfg.C_h,
                    C_z=cfg.C_z,
                    dilations=cfg.dilations,
                ).to(cfg.device)
                optimizer = optim.Adam(model.parameters(), lr=cfg.lr)

                best_val = float('inf')
                best_path = save_dir / 'best.pt'

                for epoch in range(1, cfg.epochs + 1):
                    print(f"
Epoch {epoch}/{cfg.epochs}")
                    train_metrics = train_epoch(model, train_loader, optimizer, cfg, epoch)
                    val_metrics = validate(model, val_loader, cfg)

                    train_recon = train_metrics['recon_lf'] + train_metrics['recon_bp'] + train_metrics['recon_hf']
                    train_kl = train_metrics['kl_lf'] + train_metrics['kl_bp'] + train_metrics['kl_hf']
                    val_recon = val_metrics['recon_lf'] + val_metrics['recon_bp'] + val_metrics['recon_hf']
                    val_kl = val_metrics['kl_lf'] + val_metrics['kl_bp'] + val_metrics['kl_hf']

                    print(
                        f"  Train Total={train_metrics['total']:.4f} Recon={train_recon:.4f} KL={train_kl:.4f}
"
                        f"  Val   Total={val_metrics['total']:.4f} Recon={val_recon:.4f} KL={val_kl:.4f}"
                    )

                    if val_metrics['total'] < best_val:
                        best_val = val_metrics['total']
                        torch.save(
                            {
                                'epoch': epoch,
                                'model_state_dict': model.state_dict(),
                                'optimizer_state_dict': optimizer.state_dict(),
                                'val_metrics': val_metrics,
                                'config': cfg,
                            },
                            best_path,
                        )
                        print(f"  Saved new best checkpoint -> {best_path.name}")

                    ckpt_path = save_dir / f'checkpoint_epoch{epoch:02d}.pt'
                    torch.save(
                        {
                            'epoch': epoch,
                            'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(),
                            'val_metrics': val_metrics,
                            'config': cfg,
                        },
                        ckpt_path,
                    )

                print(f"
Best validation loss: {best_val:.4f}")
                return model


In [ ]:
trained_model = run_training(config)